In [10]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import re
from pathlib import Path

In [11]:
BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"

In [12]:
def fetch_all_verslagen():
    rows = []
    skip = 0

    while True:
        params = {
            "$filter": "Verwijderd eq false",
            "$select": "Id,Soort,Status,Vergadering_Id",
            "$top": 250,
            "$skip": skip
        }

        r = requests.get(f"{BASE}/Verslag", params=params, timeout=60)
        r.raise_for_status()
        batch = r.json().get("value", [])

        if not batch:
            break

        rows.extend(batch)
        skip += 250

    return pd.DataFrame(rows)

verslag_df = fetch_all_verslagen()
print(len(verslag_df))
verslag_df.head()

22654


,Id,Soort,Status,Vergadering_Id
0,94fbdc2b-fd65-4610-a340-000018d72027,Tussenpublicatie,Ongecorrigeerd,3cb69a3e-7f81-404f-bdcd-6d7e64638f54
1,d25cda29-31db-4f99-8ffc-0006a9dd9be5,Tussenpublicatie,Ongecorrigeerd,9d5e49d8-11d4-4034-a937-9cb61e22357f
2,571d1053-faaa-41f0-b61b-000fb7384c2c,Voorpublicatie,Casco,625b554a-fd34-4988-b828-3dfc0a092af4
3,3c686811-8a3d-4213-88eb-00155c753f29,Voorpublicatie,Casco,46059f48-29dc-4e72-9b3b-5df4d7487483
4,57f69332-8f44-4e5a-a6c2-0018384cd49f,Voorpublicatie,Casco,6ded85e6-6135-4920-afcd-fe12b08cf14d


In [13]:
def download_xmls(verslag_df, out_dir="verslagen_xml", limit=10):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    print("Saving to:", out.resolve())

    for i, row in verslag_df.iterrows():
        if limit is not None and i >= limit:
            break

        verslag_id = row["Id"]
        url = f"{BASE}/Verslag({verslag_id})/resource"

        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()

            filepath = out / f"{verslag_id}.xml"
            filepath.write_bytes(r.content)
            print("Saved:", filepath)

        except Exception as e:
            print(f"Failed for {verslag_id}: {e}")

In [14]:
download_xmls(verslag_df, out_dir="verslagen_xml", limit=10)

Saving to: C:\Users\Dan\Documents\GitHub\tweedekamer\verslagen_xml
Saved: verslagen_xml\94fbdc2b-fd65-4610-a340-000018d72027.xml
Saved: verslagen_xml\d25cda29-31db-4f99-8ffc-0006a9dd9be5.xml
Saved: verslagen_xml\571d1053-faaa-41f0-b61b-000fb7384c2c.xml
Saved: verslagen_xml\3c686811-8a3d-4213-88eb-00155c753f29.xml
Saved: verslagen_xml\57f69332-8f44-4e5a-a6c2-0018384cd49f.xml
Saved: verslagen_xml\b8e463d0-bccc-4663-b45e-00193d12c88b.xml
Saved: verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8be1c.xml
Saved: verslagen_xml\6109c39c-13b6-45fb-9a54-002799ae66c1.xml
Saved: verslagen_xml\530ea904-569c-4ce0-a829-0028ddaa3618.xml
Saved: verslagen_xml\bd44a3cf-a67e-4b31-bebc-002d09ca7c83.xml


In [15]:
list(Path("verslagen_xml").glob("*.xml"))[:5]

[WindowsPath('verslagen_xml/3c686811-8a3d-4213-88eb-00155c753f29.xml'),
 WindowsPath('verslagen_xml/4a7ea7c4-40b2-46b9-9cfe-002515f8be1c.xml'),
 WindowsPath('verslagen_xml/530ea904-569c-4ce0-a829-0028ddaa3618.xml'),
 WindowsPath('verslagen_xml/571d1053-faaa-41f0-b61b-000fb7384c2c.xml'),
 WindowsPath('verslagen_xml/57f69332-8f44-4e5a-a6c2-0018384cd49f.xml')]

In [16]:
NS = {"tk": "http://www.tweedekamer.nl/ggm/vergaderverslag/v1.0"}

def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()

def extract_text_from_node(node):
    if node is None:
        return ""
    parts = []
    for t in node.itertext():
        t = clean_text(t)
        if t:
            parts.append(t)
    return clean_text(" ".join(parts))

def parse_speeches_from_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []

    vergadering = root.find("tk:vergadering", NS)
    vergadering_id = vergadering.attrib.get("objectid", "") if vergadering is not None else ""
    vergadering_titel = vergadering.findtext("tk:titel", default="", namespaces=NS) if vergadering is not None else ""
    vergadering_datum = vergadering.findtext("tk:datum", default="", namespaces=NS) if vergadering is not None else ""

    for activiteit in root.findall(".//tk:activiteit", NS):
        activiteit_id = activiteit.attrib.get("objectid", "")
        activiteit_soort = activiteit.attrib.get("soort", "")
        activiteit_titel = activiteit.findtext("tk:titel", default="", namespaces=NS)
        activiteit_onderwerp = activiteit.findtext("tk:onderwerp", default="", namespaces=NS)

        for activiteitdeel in activiteit.findall(".//tk:activiteitdeel[@soort='Spreekbeurt']", NS):
            spreekbeurt_id = activiteitdeel.attrib.get("objectid", "")

            for woordvoerder in activiteitdeel.findall(".//tk:woordvoerder", NS):
                spreker = woordvoerder.find("tk:spreker", NS)

                speaker_id = spreker.attrib.get("objectid", "") if spreker is not None else ""
                speaker_name = spreker.findtext("tk:weergavenaam", default="", namespaces=NS) if spreker is not None else ""
                speaker_party = spreker.findtext("tk:fractie", default="", namespaces=NS) if spreker is not None else ""
                speaker_role = spreker.findtext("tk:functie", default="", namespaces=NS) if spreker is not None else ""

                speech_begin = woordvoerder.findtext("tk:markeertijdbegin", default="", namespaces=NS)
                speech_end = woordvoerder.findtext("tk:markeertijdeind", default="", namespaces=NS)
                is_voorzitter = woordvoerder.findtext("tk:isvoorzitter", default="", namespaces=NS)

                tekst_node = woordvoerder.find("tk:tekst", NS)
                speech_text = extract_text_from_node(tekst_node)

                if speech_text:
                    rows.append({
                        "speech_id": spreekbeurt_id,
                        "vergadering_id": vergadering_id,
                        "vergadering_titel": vergadering_titel,
                        "vergadering_datum": vergadering_datum,
                        "activiteit_id": activiteit_id,
                        "activiteit_soort": activiteit_soort,
                        "activiteit_titel": activiteit_titel,
                        "activiteit_onderwerp": activiteit_onderwerp,
                        "speaker_id": speaker_id,
                        "speaker_name": speaker_name,
                        "speaker_party": speaker_party,
                        "speaker_role": speaker_role,
                        "speech_begin": speech_begin,
                        "speech_end": speech_end,
                        "is_voorzitter": is_voorzitter,
                        "speech_text": speech_text,
                        "source_xml": str(xml_path)
                    })

    return pd.DataFrame(rows)

In [17]:
def parse_all_xmls(xml_dir="verslagen_xml"):
    frames = []

    for xml_file in Path(xml_dir).glob("*.xml"):
        try:
            df = parse_speeches_from_xml(xml_file)
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f"Error in {xml_file}: {e}")

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

speeches_df = parse_all_xmls("verslagen_xml")
print(len(speeches_df))
speeches_df.head()

190


,speech_id,vergadering_id,vergadering_titel,vergadering_datum,activiteit_id,activiteit_soort,activiteit_titel,activiteit_onderwerp,speaker_id,speaker_name,speaker_party,speaker_role,speech_begin,speech_end,is_voorzitter,speech_text,source_xml
0,cff6b424-487b-4c55-8095-762bdc20f8e2,d5f8a00e-b29d-43de-9a10-e3483b15b98f,Evaluatie Wet hervorming herziening ten voordele,2019-09-12T00:00:00,2ffc8ff6-1d22-4c39-88e4-2b96aa8a6e0d,Algemeen overleg,Evaluatie Wet hervorming herziening ten voordele,Evaluatie Wet hervorming herziening ten voordele,6c78e05a-8239-4c47-bf4c-b66d904b777e,Wijngaarden van,VVD,lid Tweede Kamer,2019-09-12T10:00:04,2019-09-12T10:01:05,true,De voorzitter : Goedemorgen allemaal. Welkom i...,verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8...
1,b3db3646-df51-4877-9ac9-e338325c2b0b,d5f8a00e-b29d-43de-9a10-e3483b15b98f,Evaluatie Wet hervorming herziening ten voordele,2019-09-12T00:00:00,2ffc8ff6-1d22-4c39-88e4-2b96aa8a6e0d,Algemeen overleg,Evaluatie Wet hervorming herziening ten voordele,Evaluatie Wet hervorming herziening ten voordele,bb923669-1672-41e8-a9e8-cbfa4b82b217,Groothuizen,D66,lid Tweede Kamer,2019-09-12T10:01:05,2019-09-12T10:01:12,false,"De heer Groothuizen (D66): Voorzitter, wilt u ...",verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8...
2,b3db3646-df51-4877-9ac9-e338325c2b0b,d5f8a00e-b29d-43de-9a10-e3483b15b98f,Evaluatie Wet hervorming herziening ten voordele,2019-09-12T00:00:00,2ffc8ff6-1d22-4c39-88e4-2b96aa8a6e0d,Algemeen overleg,Evaluatie Wet hervorming herziening ten voordele,Evaluatie Wet hervorming herziening ten voordele,bb923669-1672-41e8-a9e8-cbfa4b82b217,Groothuizen,D66,lid Tweede Kamer,2019-09-12T10:01:13,2019-09-12T10:05:37,false,De heer Groothuizen (D66): Dat heeft ook mijn ...,verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8...
3,b3db3646-df51-4877-9ac9-e338325c2b0b,d5f8a00e-b29d-43de-9a10-e3483b15b98f,Evaluatie Wet hervorming herziening ten voordele,2019-09-12T00:00:00,2ffc8ff6-1d22-4c39-88e4-2b96aa8a6e0d,Algemeen overleg,Evaluatie Wet hervorming herziening ten voordele,Evaluatie Wet hervorming herziening ten voordele,bb923669-1672-41e8-a9e8-cbfa4b82b217,Groothuizen,D66,lid Tweede Kamer,2019-09-12T10:06:15,2019-09-12T10:06:57,false,De heer Groothuizen (D66): Dat punt is inderda...,verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8...
4,737eda4c-861f-4925-8514-88ee0c2a4f2a,d5f8a00e-b29d-43de-9a10-e3483b15b98f,Evaluatie Wet hervorming herziening ten voordele,2019-09-12T00:00:00,2ffc8ff6-1d22-4c39-88e4-2b96aa8a6e0d,Algemeen overleg,Evaluatie Wet hervorming herziening ten voordele,Evaluatie Wet hervorming herziening ten voordele,d95bda8c-c94d-4fd9-b6a0-db9e60aa2b59,Nispen van,SP,lid Tweede Kamer,2019-09-12T10:07:38,2019-09-12T10:12:34,false,De heer Van Nispen (SP): Geen onschuldigen in ...,verslagen_xml\4a7ea7c4-40b2-46b9-9cfe-002515f8...


In [18]:
speeches_df.to_csv("speeches_raw.csv", index=False)

In [19]:
SCHEMA = {
    "speech_id": "string",
    "vergadering_id": "string",
    "activiteit_id": "string",
    "agendapunt_id": "string",
    "motion_id": "string",
    "politician_id": "string",
    "politician_name": "string",
    "party": "string",
    "role": "string",
    "session_type": "string",
    "session_title": "string",
    "topic_title": "string",
    "topic_subject": "string",
    "speech_begin": "datetime64[ns]",
    "speech_end": "datetime64[ns]",
    "speech_duration_seconds": "float",
    "full_speech_text": "string",
    "motion_passed": "int8",
    "motion_outcome_raw": "string",
    "day_of_week": "int8",
    "hour_of_day": "int8",
    "time_bin": "category",
    "month": "int8",
    "is_weekend": "int8",
    "sentiment_label": "string",
    "sentiment_score_neg": "float32",
    "sentiment_score_neu": "float32",
    "sentiment_score_pos": "float32",
}

In [20]:
# src/config.py
BASE_URL = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"
TOP = 250
RAW_XML_DIR = "data/raw/xml"
RAW_META_DIR = "data/raw/metadata"
PROCESSED_DIR = "data/processed"
RANDOM_STATE = 42
TARGET_MIN_SPEECHES = 5000
TARGET_MAX_SPEECHES = 7000

In [21]:
# src/api_client.py
import time
import requests
import pandas as pd
from typing import Dict, Optional

class TKApiClient:
    def __init__(self, base_url: str, top: int = 250, sleep_s: float = 0.2):
        self.base_url = base_url
        self.top = top
        self.sleep_s = sleep_s
        self.session = requests.Session()

    def fetch_all(self, entity: str, params: Optional[Dict[str, str]] = None) -> pd.DataFrame:
        rows = []
        skip = 0
        params = params or {}

        while True:
            query = dict(params)
            query["$top"] = self.top
            query["$skip"] = skip

            r = self.session.get(f"{self.base_url}/{entity}", params=query, timeout=60)
            r.raise_for_status()
            batch = r.json().get("value", [])

            if not batch:
                break

            rows.extend(batch)
            skip += self.top
            time.sleep(self.sleep_s)

        return pd.DataFrame(rows)

    def download_resource(self, entity: str, entity_id: str) -> bytes:
        r = self.session.get(f"{self.base_url}/{entity}({entity_id})/resource", timeout=120)
        r.raise_for_status()
        return r.content

In [22]:
# src/api_client.py
import time
import requests
import pandas as pd
from typing import Dict, Optional

class TKApiClient:
    def __init__(self, base_url: str, top: int = 250, sleep_s: float = 0.2):
        self.base_url = base_url
        self.top = top
        self.sleep_s = sleep_s
        self.session = requests.Session()

    def fetch_all(self, entity: str, params: Optional[Dict[str, str]] = None) -> pd.DataFrame:
        rows = []
        skip = 0
        params = params or {}

        while True:
            query = dict(params)
            query["$top"] = self.top
            query["$skip"] = skip

            r = self.session.get(f"{self.base_url}/{entity}", params=query, timeout=60)
            r.raise_for_status()
            batch = r.json().get("value", [])

            if not batch:
                break

            rows.extend(batch)
            skip += self.top
            time.sleep(self.sleep_s)

        return pd.DataFrame(rows)

    def download_resource(self, entity: str, entity_id: str) -> bytes:
        r = self.session.get(f"{self.base_url}/{entity}({entity_id})/resource", timeout=120)
        r.raise_for_status()
        return r.content

In [23]:
# src/parse_xml.py
import re
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd

NS = {"tk": "http://www.tweedekamer.nl/ggm/vergaderverslag/v1.0"}

def clean_text(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()

def extract_text_from_node(node) -> str:
    if node is None:
        return ""
    parts = []
    for t in node.itertext():
        t = clean_text(t)
        if t:
            parts.append(t)
    return clean_text(" ".join(parts))

def parse_speeches_from_xml(xml_path: str | Path) -> pd.DataFrame:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    rows = []

    vergadering = root.find("tk:vergadering", NS)
    vergadering_id = vergadering.attrib.get("objectid", "") if vergadering is not None else ""
    vergadering_soort = vergadering.attrib.get("soort", "") if vergadering is not None else ""
    vergadering_titel = vergadering.findtext("tk:titel", default="", namespaces=NS) if vergadering is not None else ""
    vergadering_datum = vergadering.findtext("tk:datum", default="", namespaces=NS) if vergadering is not None else ""

    for activiteit in root.findall(".//tk:activiteit", NS):
        activiteit_id = activiteit.attrib.get("objectid", "")
        activiteit_soort = activiteit.attrib.get("soort", "")
        activiteit_titel = activiteit.findtext("tk:titel", default="", namespaces=NS)
        activiteit_onderwerp = activiteit.findtext("tk:onderwerp", default="", namespaces=NS)

        for activiteitdeel in activiteit.findall(".//tk:activiteitdeel[@soort='Spreekbeurt']", NS):
            spreekbeurt_id = activiteitdeel.attrib.get("objectid", "")
            spreekbeurt_titel = activiteitdeel.findtext("tk:titel", default="", namespaces=NS)

            for woordvoerder in activiteitdeel.findall(".//tk:woordvoerder", NS):
                spreker = woordvoerder.find("tk:spreker", NS)

                speaker_id = spreker.attrib.get("objectid", "") if spreker is not None else ""
                speaker_name = spreker.findtext("tk:weergavenaam", default="", namespaces=NS) if spreker is not None else ""
                speaker_party = spreker.findtext("tk:fractie", default="", namespaces=NS) if spreker is not None else ""
                speaker_role = spreker.findtext("tk:functie", default="", namespaces=NS) if spreker is not None else ""

                speech_begin = woordvoerder.findtext("tk:markeertijdbegin", default="", namespaces=NS)
                speech_end = woordvoerder.findtext("tk:markeertijdeind", default="", namespaces=NS)
                is_voorzitter = woordvoerder.findtext("tk:isvoorzitter", default="", namespaces=NS)

                text_node = woordvoerder.find("tk:tekst", NS)
                speech_text = extract_text_from_node(text_node)

                if speech_text:
                    rows.append({
                        "speech_id": spreekbeurt_id,
                        "vergadering_id": vergadering_id,
                        "session_title": vergadering_titel,
                        "session_date": vergadering_datum,
                        "session_type": activiteit_soort,
                        "activiteit_id": activiteit_id,
                        "topic_title": activiteit_titel,
                        "topic_subject": activiteit_onderwerp,
                        "politician_id": speaker_id,
                        "politician_name": speaker_name,
                        "party": speaker_party,
                        "role": speaker_role,
                        "speech_begin": speech_begin,
                        "speech_end": speech_end,
                        "is_voorzitter": is_voorzitter,
                        "full_speech_text": speech_text,
                        "source_xml": str(xml_path),
                    })

    return pd.DataFrame(rows)

def parse_all_xmls(xml_dir: str | Path) -> pd.DataFrame:
    frames = []
    for fp in Path(xml_dir).glob("*.xml"):
        try:
            df = parse_speeches_from_xml(fp)
            if not df.empty:
                frames.append(df)
        except Exception as e:
            print(f"Failed parsing {fp}: {e}")
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [24]:
list(Path("verslagen_xml").glob("*.xml"))


[WindowsPath('verslagen_xml/3c686811-8a3d-4213-88eb-00155c753f29.xml'),
 WindowsPath('verslagen_xml/4a7ea7c4-40b2-46b9-9cfe-002515f8be1c.xml'),
 WindowsPath('verslagen_xml/530ea904-569c-4ce0-a829-0028ddaa3618.xml'),
 WindowsPath('verslagen_xml/571d1053-faaa-41f0-b61b-000fb7384c2c.xml'),
 WindowsPath('verslagen_xml/57f69332-8f44-4e5a-a6c2-0018384cd49f.xml'),
 WindowsPath('verslagen_xml/6109c39c-13b6-45fb-9a54-002799ae66c1.xml'),
 WindowsPath('verslagen_xml/94fbdc2b-fd65-4610-a340-000018d72027.xml'),
 WindowsPath('verslagen_xml/b8e463d0-bccc-4663-b45e-00193d12c88b.xml'),
 WindowsPath('verslagen_xml/bd44a3cf-a67e-4b31-bebc-002d09ca7c83.xml'),
 WindowsPath('verslagen_xml/d25cda29-31db-4f99-8ffc-0006a9dd9be5.xml')]

In [25]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print("Does local verslagen_xml exist?", Path("verslagen_xml").exists())

Current working directory: c:\Users\Dan\Documents\GitHub\tweedekamer
Does local verslagen_xml exist? True


In [28]:
from pathlib import Path

for f in ["speeches_raw.csv", "agendapunt.csv", "activiteit.csv", "besluit.csv"]:
    print(f, Path(f).exists())

speeches_raw.csv True
agendapunt.csv False
activiteit.csv False
besluit.csv False


In [27]:
# src/build_dataset.py
import pandas as pd
from pathlib import Path

def main():
    speeches = pd.read_csv("speeches_raw.csv")
    ag = pd.read_csv("data/raw/metadata/agendapunt.csv")
    act = pd.read_csv("data/raw/metadata/activiteit.csv")
    besluit = pd.read_csv("data/raw/metadata/besluit.csv")

    besluit["motion_passed"] = (besluit["BesluitSoort"] == "Stemmen - aangenomen").astype(int)
    besluit = besluit.rename(columns={"Id": "motion_id", "Agendapunt_Id": "agendapunt_id", "BesluitSoort": "motion_outcome_raw"})

    ag = ag.rename(columns={"Id": "agendapunt_id", "Activiteit_Id": "activiteit_id"})
    act = act.rename(columns={"Id": "activiteit_id"})

    speeches["speech_begin"] = pd.to_datetime(speeches["speech_begin"], errors="coerce")
    ag["Aanvangstijd"] = pd.to_datetime(ag["Aanvangstijd"], errors="coerce")
    ag["Eindtijd"] = pd.to_datetime(ag["Eindtijd"], errors="coerce")

    tmp = speeches.merge(ag, on="activiteit_id", how="left")
    tmp = tmp[
        (tmp["speech_begin"].isna()) |
        (
            (tmp["Aanvangstijd"].isna() | (tmp["speech_begin"] >= tmp["Aanvangstijd"])) &
            (tmp["Eindtijd"].isna() | (tmp["speech_begin"] <= tmp["Eindtijd"]))
        )
    ].copy()

    tmp["time_diff"] = (tmp["speech_begin"] - tmp["Aanvangstijd"]).abs().dt.total_seconds()
    tmp = tmp.sort_values(["speech_id", "time_diff"]).drop_duplicates("speech_id")

    df = (
        tmp.merge(act, on="activiteit_id", how="left")
           .merge(besluit[["motion_id", "agendapunt_id", "motion_outcome_raw", "motion_passed"]], on="agendapunt_id", how="left")
    )

    Path("data/processed").mkdir(parents=True, exist_ok=True)
    df.to_csv("data/processed/speeches_joined.csv", index=False)

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/metadata/agendapunt.csv'

In [ ]:
# src/preprocess.py
import re
import numpy as np
import pandas as pd

def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def strip_leading_speaker_label(text: str) -> str:
    return re.sub(r"^(De heer|Mevrouw|Minister|Staatssecretaris|De voorzitter).*?:\s*", "", text).strip()

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df["speech_begin"] = pd.to_datetime(df["speech_begin"], errors="coerce")
    df["speech_end"] = pd.to_datetime(df["speech_end"], errors="coerce")
    df["speech_duration_seconds"] = (df["speech_end"] - df["speech_begin"]).dt.total_seconds()

    df["hour_of_day"] = df["speech_begin"].dt.hour
    df["day_of_week"] = df["speech_begin"].dt.dayofweek
    df["month"] = df["speech_begin"].dt.month
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    def time_bin(h):
        if pd.isna(h):
            return "unknown"
        if 6 <= h < 12:
            return "morning"
        if 12 <= h < 18:
            return "afternoon"
        if 18 <= h < 24:
            return "evening"
        return "night"

    df["time_bin"] = df["hour_of_day"].apply(time_bin)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)
    return df

def main():
    df = pd.read_csv("data/processed/speeches_joined.csv")
    df = df[df["is_voorzitter"] != "true"].copy()
    df["full_speech_text"] = df["full_speech_text"].map(normalize_text).map(strip_leading_speaker_label)
    df = df[df["full_speech_text"].str.len() > 30].copy()
    df = add_time_features(df)

    labeled = df[df["motion_passed"].notna()].copy()
    labeled = labeled.sample(n=min(len(labeled), 6500), random_state=42)

    labeled.to_csv("data/processed/speeches_modeling.csv", index=False)

if __name__ == "__main__":
    main()

In [ ]:
df = pd.read_csv("data/processed/speeches_modeling.csv")
df = df.sort_values("speech_begin")

train_end = int(len(df) * 0.70)
val_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [ ]:
# src/label_sentiment.py
import pandas as pd
from transformers import pipeline

MODEL_NAME = "DTAI-KULeuven/robbert-v2-dutch-sentiment"

def main():
    df = pd.read_csv("data/processed/speeches_modeling.csv")

    clf = pipeline(
        "text-classification",
        model=MODEL_NAME,
        tokenizer=MODEL_NAME,
        truncation=True,
        top_k=None
    )

    outputs = clf(df["full_speech_text"].tolist(), batch_size=16)

    labels, negs, neus, poss = [], [], [], []

    for out in outputs:
        score_map = {d["label"].lower(): d["score"] for d in out}
        neg = score_map.get("negative", 0.0)
        neu = score_map.get("neutral", 0.0)
        pos = score_map.get("positive", 0.0)

        best = max(
            [("negative", neg), ("neutral", neu), ("positive", pos)],
            key=lambda x: x[1]
        )[0]

        labels.append(best)
        negs.append(neg)
        neus.append(neu)
        poss.append(pos)

    df["sentiment_label"] = labels
    df["sentiment_score_neg"] = negs
    df["sentiment_score_neu"] = neus
    df["sentiment_score_pos"] = poss

    df.to_csv("data/processed/speeches_labeled.csv", index=False)

if __name__ == "__main__":
    main()